# 自然语言处理

## 本节内容概览
- 自然语言和单词的分布式表示
    - 共现矩阵
    - 余弦相似度
- word2vec 及改进
- 循环神经网络 RNN
- Ganted RNN 框架： LSTM  
- 基于 RNN 生成文本

## 什么是自然语言处理（Natural Language Processing， NLP）

人类平常使用的语言，中文、英语等被称为自然语言。自然语言处理（Natural Language Processing， NLP）就是处理自然语言的科学。

例如搜索引擎，机器翻译，问答系统，自动文本摘要，文本情感分析等。

语言的含义由文字或者说单词组成，单词是语言中具备含义的最小单位，为了让计算机理解自然语言，让它理解单词是首要的事情。

![05-nlp.png](source/05-nlp.png)


## 0. 理解单词含义

首先讨论一些巧妙蕴含了单词含义的表示方法：
- 基于人工整理好的同义词词典的方法
- 基于统计信息表示单词的方法（计数方法）
- 利用神经网络的基于推理的方法（word2vec 方法）

### 0.0 基于同义词词典的方法

要表示单词含义，首先可以考虑的是通过人工方式来定义单词。例如《新华字典》，逐词说明每一个单词的含义。

在自然语言历史上，已经有过许多次尝试，目前广泛使用的不是《新华字典》这类常规词典，而是一种被称为**同义词词典（thesaurus）**的词典，同义词或近义词被归类到同一个组别。

例如，car 的同义词有 automobile、motorcar 等。

此外，自然语言处理中也会使用更加细粒度的关系。例如“上位-下位”，“整体-部分”关系。

总之，利用对所有单词创建近义词集合，并用图表示单词之间的关系，就可以将单词含义直接或间接的教给计算机。

![05-hypernym–hyponym-relations.png](source/05-hypernym–hyponym-relations.png)

### 0.0.1 WordNet

自然语言处理领域，最著名的同义词词典是 WordNet。WordNet 是普林斯顿大学于 1985 年开始开发的同义词词典，迄今已用于许多研究，并活跃于各种自然语言处理应用中。
使用 WordNet，可以获得单词的近义词，或者利用单词网络。使用单词网络，可以计算单词之间的相似度。

### 0.0.2 同义词词典的问题

WordNet 等同义词词典中对大量单词定义了同义词和层级结构关系等。 利用这些知识，可以（间接地）让计算机理解单词含义。不过，人工标记也存在一些较大的缺陷。

- 难以顺应时代变化。随着时间推移，新词不断出现，旧词久而废止。此外，语言的含义也会随着时间推移而变化。例如“打卡”：过去主要指上下班签到，现在还可以指“到某地游玩并拍照记录”。要处理这种单词变化，就要人工不断更新同义词词典。
- 无法表示单词的微妙差异，例如“观看电影”，“查看信息”，两个动词难以互换。
- 人力成本高，维护困难。

*不仅限于自然语言处理，在图像识别领域，多年来也一直是人工设计特征量。但是，随着深度学习的出现，现在从原始图像直接获得最终结果已成为可能，人为介入的必要性大幅降低。在自然语言处理领域也有类似现象。也就是说，我们正在从人工制作词典或设计特征量的旧范式，向尽量减少人为干预的、仅从文本数据中获取最终结果的新范式转移。*

### 0.1 基于计数的方法

从介绍基于计数的方法开始，我们将使用语料库（corpus）。简而言之，语料库就是大量的文本数据。不过，语料库并不是胡乱收集数据，一般收集的都是用于自然语言处理研究和应用的文本数据。

*自然语言处理领域中使用的语料库有时会给文本数据添加额外的 信息。比如，可以给文本数据的各个单词标记词性。在这种情况下，为了方便计算机处理，语料库通常会被结构化（比如，采用树结构等数据形式）。这里，假定我们使用的语料库没有添加标签， 而是作为一个大的文本文件，只包含简单的文本数据。*

### 0.1.1 基于 Python 的语料库的处理

现在通过 Python 的交互模式，对一个非常小的文本数据（语料库）进行预处理。这里的预处理指的是，将文本分割为单词（分词），并将分割后的单词列表转化为单词 ID 列表。

In [2]:
import numpy as np

TEXT = 'You say goodbye and I say hello.'

In [13]:
def proprecess(text: str):
    '''
    proprecess
    '''
    text = text.lower()
    text = text.replace('.', ' .')
    words = text.split(' ')
    print(words)

    word_to_id = {}
    id_to_word = {}

    for word in words:
        if word not in word_to_id:
            new_id = len(word_to_id)
            word_to_id[word] = new_id
            id_to_word[new_id] = word

    corpus = np.array([word_to_id[w] for w in words])

    return corpus, word_to_id, id_to_word

corpus, word_to_id, id_to_word = proprecess(TEXT)

print(corpus)
print(word_to_id)
print(id_to_word)


['you', 'say', 'goodbye', 'and', 'i', 'say', 'hello', '.']
[0 1 2 3 4 1 5 6]
{'you': 0, 'say': 1, 'goodbye': 2, 'and': 3, 'i': 4, 'hello': 5, '.': 6}
{0: 'you', 1: 'say', 2: 'goodbye', 3: 'and', 4: 'i', 5: 'hello', 6: '.'}


### 0.1.2 分布式假设

在自然语言处理的历史中，用向量表示单词的研究有很多。如果仔细看一下这些研究，就会发现几乎所有的重要方法都基于一个简单的想法，这个想法就是“某个单词的含义由它周围的单词形成”，这称为分布式假设（distributional hypothesis）。

“上下文”指某个居中单词的周围词汇。在此，我们将上下文大小（即周围单词的个数）称为窗口大小（window size）。例如，窗口大小为 2，上下文包含左右各 2。个单词。

![alt text](source/05-context.png)

### 0.1.3 共现矩阵

若要基于分布式假设使用向量表示单词，最直接的方法是对一个单词周围单词的数量进行计数。

共现矩阵是一种用来表示词语之间共同出现关系的矩阵。它统计一个词在一定窗口范围内与其他词共同出现的次数，次数越多，说明两个词在语料中的关联程度越高。

In [ ]:
def create_co_matrix(corpus, vocab_size, window_size = 1):
    '''
    create comatrix
    '''
    corpus_size = len(corpus)
    corpus_matrix = np.zeros(shape=(vocab_size, vocab_size), dtype=np.int32) # 创建 vocabe_size * vocabe_size 尺寸的共现矩阵

    for index, word_id in enumerate(corpus):
        for i in range(1, window_size + 1):
            left_index = index - i
            right_index = index + i

            if left_index >= 0:
                left_word_id = corpus[left_index]
                corpus_matrix[word_id, left_word_id] += 1
            
            if right_index < corpus_size:
                right_word_id = corpus[right_index]
                corpus_matrix[word_id, right_word_id] += 1
    return corpus_matrix

### 0.1.4 向量相似度

在测量单词表示相似度方面，余弦相似度很常用。

In [ ]:
# 余弦相似度计算
def cos_similarity(x, y, eps = 1e-8):
    '''
    cos similarity
    '''
    nx = x / np.sqrt(np.sum(x ** 2) + eps) # x 正规化
    ny = y / np.sqrt(np.sum(y ** 2) + eps) # y 正规化
    # eps 防止分母为零
    return np.dot(nx, ny)

In [16]:
corpus, word_to_id, id_to_word = proprecess(TEXT)
print(corpus, word_to_id, id_to_word)
C = create_co_matrix(corpus, len(word_to_id), window_size=1)
c0 = C[word_to_id['you']]
c1 = C[word_to_id['i']]

similarity = cos_similarity(c0, c1)
print(similarity)

['you', 'say', 'goodbye', 'and', 'i', 'say', 'hello', '.']
[0 1 2 3 4 1 5 6] {'you': 0, 'say': 1, 'goodbye': 2, 'and': 3, 'i': 4, 'hello': 5, '.': 6} {0: 'you', 1: 'say', 2: 'goodbye', 3: 'and', 4: 'i', 5: 'hello', 6: '.'}
0.7071067758832467


在实现余弦相似度的基础上，可以在某个单词作为查询词时，将与该单词相似的单词按降序显示。

In [ ]:
def most_similar(query, word_to_id, id_to_word, word_matrix, top=5):
    '''
    most similar
    '''

    # 取出查询词
    if query not in word_to_id:
        print(f"{query} is not found")
        return

    print(f"\n[query] {query}")
    query_id = word_to_id[query]
    query_vec = word_matrix[query_id]

    # 计算余弦相似度
    vocab_size = len(id_to_word)
    similarity = np.zeros(vocab_size)

    for i in range(vocab_size):
        similarity[i] = cos_similarity(word_matrix[i], query_vec)

    # 基于余弦相似度，按降序输出值
    count = 0
    for i in (-1 * similarity).argsort():
        # 对于 `argsort` 函数，可以按升序排序传入的数组，但是返回值是数组的索引。
        # np.array([100, -20, 2]).argsort(), 排序的结果为 array(1,2,0)
        if id_to_word[i] == query:
            continue
        print(f"{id_to_word[i]}: {similarity[i]}")

        count += 1
        if count >= top:
            return